## EDA with DuckDB

In [70]:
from pathlib import Path
import duckdb
import requests
from tqdm import tqdm
import pandas as pd

### Defile the paths
The deafult option is set to "Books" categoty 

In [71]:
CATEGORY = "Appliances"
BASE_URL = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw"
REVIEWS_URL = f"{BASE_URL}/review_categories/{CATEGORY}.jsonl.gz"
META_URL    = f"{BASE_URL}/meta_categories/meta_{CATEGORY}.jsonl.gz"
DATA_DIR = "../data/processed"
RAW_DIR = "../data/raw"

In [72]:
print(REVIEWS_URL)

https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Appliances.jsonl.gz


### Initialize an in-memory DB connection

In [73]:
c2 = duckdb.connect()

### Lets open the file online and see the first few lines

In [74]:
head_reviews = c2.execute(f"SELECT * FROM read_json_auto('{REVIEWS_URL}') LIMIT 5").df()
head_reviews

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,5.0,Work great,work great. use a new one every month,[],B01N0TQ0OH,B01N0TQ0OH,AGKHLEW2SOWHNMFQIJGBECAF7INQ,1519317108692,0,True
1,5.0,excellent product,Little on the thin side,[],B07DD2DMXB,B07DD37QPZ,AHWWLSPCJMALVHDDVSUGICL6RUCA,1664746863446,0,True
2,5.0,Happy customer!,"Quick delivery, fixed the issue!",[],B082W3Z9YK,B082W3Z9YK,AHZIJGKEWRTAEOZ673G5B3SNXEGQ,1607225435363,0,True
3,5.0,Amazing value,I wasn't sure whether these were worth it or n...,[],B078W2BJY8,B078W2BJY8,AFGUPTDFAWOHHL4LZDV27ERDNOYQ,1534104184306,0,True
4,5.0,Dryer parts,Easy to install got the product expected to re...,[],B08C9LPCQV,B08C9LPCQV,AELFJFAXQERUSMTXJQ6SYFFRDWMA,1620176603754,0,True


In [75]:
# executes right over the internet -- i just want five rows to preview so doesn't take long
head_meta = c2.execute(f"SELECT * FROM read_json_auto('{META_URL}') LIMIT 5").df()
head_meta

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together
0,Industrial & Scientific,"ROVSUN Ice Maker Machine Countertop, Make 44lb...",3.7,61,[【Quick Ice Making】This countertop ice machine...,[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Our Point of View on the Euhomy Ic...,ROVSUN,"[Appliances, Refrigerators, Freezers & Ice Mak...","{'Brand': '""ROVSUN""', 'Model Name': '""ICM-2005...",B08Z743RRD,None
1,Tools & Home Improvement,"HANSGO Egg Holder for Refrigerator, Deviled Eg...",4.2,75,"[Plastic, Practical Kitchen Storage - Our egg ...",[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': '10 Eggs Egg Holder for Refrigerato...,HANSGO,"[Appliances, Parts & Accessories, Refrigerator...","{'Manufacturer': '""HANSGO""', 'Part Number': '""...",B097BQDGHJ,None
2,Tools & Home Improvement,"Clothes Dryer Drum Slide, General Electric, Ho...",3.5,18,[],"[Brand new dryer drum slide, replaces General ...",NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],GE,"[Appliances, Parts & Accessories]","{'Manufacturer': '""RPI""', 'Part Number': '""WE1...",B00IN9AGAE,None
3,Tools & Home Improvement,154567702 Dishwasher Lower Wash Arm Assembly f...,4.5,26,[MODEL NUMBER:154567702 Dishwasher Lower Wash ...,[MODEL NUMBER:154567702 Dishwasher Lower Wash ...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],folosem,"[Appliances, Parts & Accessories, Dryer Parts ...","{'Manufacturer': '""folosem""', 'Part Number': '...",B0C7K98JZS,None
4,Tools & Home Improvement,Whirlpool W10918546 Igniter,3.8,12,[This is a Genuine OEM Replacement Part.],[Whirlpool Igniter],25.07,[{'thumb': 'https://m.media-amazon.com/images/...,[],Whirlpool,"[Appliances, Parts & Accessories]","{'Manufacturer': '""Whirlpool""', 'Part Number':...",B07QZHQTVJ,None


### Download and convert to parquet on the fly

So lets download and covert to parquet on the fly (remember, we pay conversion tax once and reuse for all consequent queries -- whether in duckdb or pandas ...). 

Even if we are performing parallel processing, we need full download here, so it will take time, depending on your connection + conversion overhead.

- Downloading only 20k (`LIMIT 20000`) for a quick inspection. 
- It take a few seconds to download (will depend on your network connection)  

In [76]:
c2.execute(
    f"""
      COPY (SELECT * FROM read_json_auto('{REVIEWS_URL}')  LIMIT 20000)
      TO '{RAW_DIR}/reviews_raw.parquet'
      (FORMAT PARQUET, COMPRESSION ZSTD)
  """
)

In [77]:
c2.execute(
    f"""
      COPY (SELECT * FROM read_json_auto('{META_URL}') LIMIT 20000)
      TO '{RAW_DIR}/meta_raw.parquet'
      (FORMAT PARQUET, COMPRESSION ZSTD)
  """
)

### Merging the two files by joining on `parent_asin`

In [78]:
c2.execute(
    f"""
    COPY (
        SELECT r.*, m.title AS product_title, m.price,
                    m.average_rating, m.main_category, m.store
        FROM read_parquet('{RAW_DIR}/reviews_raw.parquet') r
        LEFT JOIN read_parquet('{RAW_DIR}/meta_raw.parquet') m USING (parent_asin)
    )
    TO '{DATA_DIR}/merged.parquet' (FORMAT PARQUET, COMPRESSION ZSTD)
"""
)

### See it as a dataframe again

In [79]:
c2.execute(f"SELECT * FROM read_parquet('{DATA_DIR}/merged.parquet')").df()


,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,product_title,price,average_rating,main_category,store
0,5.0,excellent product,Little on the thin side,[],B07DD2DMXB,B07DD37QPZ,AHWWLSPCJMALVHDDVSUGICL6RUCA,1664746863446,0,True,Essential Values 18 Pack Compatible Replacemen...,22.99,4.4,Tools & Home Improvement,Essential Values
1,5.0,Happy customer!,"Quick delivery, fixed the issue!",[],B082W3Z9YK,B082W3Z9YK,AHZIJGKEWRTAEOZ673G5B3SNXEGQ,1607225435363,0,True,279838 Dryer Heating Element by Romalon with R...,NaN,4.5,Tools & Home Improvement,Romalon
2,5.0,DO NOT purchase this ice machine.,After buying this ice machine just 15 months a...,[],B08D6RFV6D,B099ZKQJHK,AEUH4EH6XHROLT7UZPUYU2YKTYMA,1663078878875,0,True,COOLLIFE Compact Countertop Ice Maker Machine ...,NaN,4.1,Industrial & Scientific,COOLLIFE
3,2.0,They don't fit properly,Not the best quality,[],B001TH7GZA,B001TH7H0O,AHCV2CNCOCG6WECDROOUYPDZIFEQ,1610219023865,0,True,"Stanco 5557 Drip Bowl Universal, Porcelain coa...",NaN,4.3,Tools & Home Improvement,Stanco
4,5.0,so far so good,but i havent had it long a year down the road...,[],B09B21HWFM,B09W5PMK5X,AHGAOIZVODNHYMNCBV4DECZH42UQ,1650153667807,2,True,COMFEE’ Washing Machine 2.4 Cu.ft LED Portable...,399.00,3.5,Appliances,COMFEE'
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,5.0,Natural Product,We love the Natural Filters for our Chemex Pou...,[],B017OFOP68,B017OFOP68,AG3TLL5HJQ7FPK3KJCIF4VKL7TYA,1613496241659,0,True,NaN,NaN,NaN,NaN,NaN
19996,3.0,Three Stars,"Refrigerator doesn't recognize this filter, so...",[],B01I3M7K3Q,B01I3M7K3Q,AHR6M4DYJMUKR6T4VB6MFN67MEOA,1486870837000,1,True,NaN,NaN,NaN,NaN,NaN
19997,5.0,Five Stars,As described.,[],B002YJ3YRQ,B002YJ3YRQ,AHR6M4DYJMUKR6T4VB6MFN67MEOA,1477075770000,0,True,NaN,NaN,NaN,NaN,NaN
19998,5.0,Fits most makers,Used to filter a Mr. Coffee. Very good.,[],B00GPXTA0C,B00GPXTA0C,AHVZWMOITJQRTH4BVOCJXWLTFUXQ,1651336382350,0,True,NaN,NaN,NaN,NaN,NaN


### EDA example duckdb Rating distribution


In [80]:
# EDA example duckdb Rating distribution
c2.execute(f"""
    SELECT
        rating,
        COUNT(*) AS cnt,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM read_parquet('{DATA_DIR}/merged.parquet')
    GROUP BY 1
    ORDER BY 1
""").df()

,rating,cnt,pct
0,1.0,1484,7.42
1,2.0,624,3.12
2,3.0,1139,5.70
3,4.0,2304,11.52
4,5.0,14449,72.25


- You can use pandas from here
- Choose your initial sample size based on your labtops and needs.

If the resulting parquet is still too large to work with, you can sample from the initial file and create a smaller file.



### Stratified sample: 

**IF resulting parquet might still be too large, but you don't just want to take the first 20000-30000 rows**, you want to sample reviews across main dimensions: long and short reviews, different stars,

What queries do: 

(1) `labelled`
- Joins reviews with metadata on `parent_asin`.
- Adds two labels to each review:
- `rating_bucket`: based on the product’s average rating
- `len_tier`: based on review length
Filters out:
- Reviews with missing/empty text
- Products without an average rating

This is the base dataset everything else builds on.

(2) `one_per_product` $\to$ `ranked`

`one_per_product`

- Within each stratum (rating_bucket x len_tier x `verified_purchase`):
- bRank reviews per product by helpful_vote DESC
- We want to keep only the best review per product per cell
(e.g., from 500 reviews --> 1 candidate)

- Take those top reviews (`product_rank = 1`)
-Rank them across the entire cell by: `helpful_vote DESC`, `random()`

This decides which products make the final cut when more than SAMPLE_PER_STRATUM products exist in a cell.

(3) Final `SELECT` Keeps only the top `SAMPLE_PER_STRATUM` rows per cell (via `stratum_rank <= N`), drops the internal ranking columns, and writes to Parquet. The result is a balanced dataset with at most N rows per stratum cell, each row from a different product, prioritising reviews with the most helpful votes.

In [81]:
SAMPLE_PER_STRATUM = 50    # reviews to keep per stratum cell (floor guarantee)
MIN_TEXT_LEN       = 20    # drop near-empty reviews (chars)
SHORT_MAX          = 100   # short: text < SHORT_MAX chars
MEDIUM_MAX         = 500   # medium: SHORT_MAX ≤ text < MEDIUM_MAX chars
                        # long: text ≥ MEDIUM_MAX chars

OUTPUT = f'{DATA_DIR}/stratified_sample.parquet'

c2.execute(f"""
COPY (
WITH labelled AS (
    SELECT
        text, rating, verified_purchase, helpful_vote,
        parent_asin, user_id, timestamp,
        product_title, price,
        average_rating, main_category,

        CASE
            WHEN average_rating >= 4.6 THEN '4.6-5.0'
            WHEN average_rating >= 4.4 THEN '4.4-4.5'
            WHEN average_rating >= 4.1 THEN '4.1-4.3'
            WHEN average_rating >= 3.7 THEN '3.7-4.0'
            WHEN average_rating >= 3.1 THEN '3.1-3.6'
            ELSE                              '<=3.0'
        END AS rating_bucket,

        CASE
            WHEN LENGTH(text) < {SHORT_MAX}  THEN 'short'
            WHEN LENGTH(text) < {MEDIUM_MAX} THEN 'medium'
            ELSE                                    'long'
        END AS len_tier

    FROM read_parquet('{DATA_DIR}/merged.parquet')
    WHERE text IS NOT NULL
    AND LENGTH(text) >= {MIN_TEXT_LEN}
    AND average_rating IS NOT NULL
),

-- Step 1: one review per product per cell: prevents a single popular product flooding a stratum, you can redefine
one_per_product AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY rating_bucket, len_tier, verified_purchase, parent_asin
            ORDER BY helpful_vote DESC, random()
        ) AS product_rank
    FROM labelled
),

-- Step 2: rank within each stratum cell, helpful reviews first
ranked AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY rating_bucket, len_tier, verified_purchase
            ORDER BY helpful_vote DESC, random()
        ) AS stratum_rank
    FROM one_per_product
    WHERE product_rank = 1
)

SELECT * EXCLUDE (product_rank, stratum_rank)
FROM ranked
WHERE stratum_rank <= {SAMPLE_PER_STRATUM}
)
TO '{OUTPUT}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

In [82]:
c2.execute(
    f"""
SELECT *
FROM read_parquet('{OUTPUT}')
LIMIT 20
"""
).fetchdf()

,text,rating,verified_purchase,helpful_vote,parent_asin,user_id,timestamp,product_title,price,average_rating,main_category,rating_bucket,len_tier
0,There are quite a few of these icemakers being...,5.0,False,12,B0BLSHMSSS,AG375WAXLZ7PIOQKIQ6KQB4J3JVQ,1644849830972,Gevi Household Ice Maker Machine | Countertop ...,NaN,4.4,Appliances,4.4-4.5,long
1,Works everytime. Good for small spaces. Uses r...,5.0,False,8,B08FDC59PB,AGYVGLCIRBPOTCDZ5YQIBNBGLHJQ,1404419963000,Panda 110V 850W Electric Compact Portable Clot...,229.00,4.4,Appliances,4.4-4.5,long
2,"After hearing of several dryer fires, I figure...",5.0,False,7,B0BS98NCX3,AEK5RL4BLD5V4UMVB5C5EZWFWXIA,1668729546415,Sealegend Dryer Vent Cleaner Kit 7 Pack Dryer ...,28.99,4.4,Tools & Home Improvement,4.4-4.5,long
3,I've been using this dishwasher for about 8 mo...,5.0,False,7,B074WCLFPV,AEGSCA4HCE6EYJ2LKPFNUUV4KMJA,1355672355000,"SPT Countertop Dishwasher, White",NaN,4.4,Appliances,4.4-4.5,long
4,I have a 6 Cup Chemex coffee maker and dont ge...,5.0,False,7,B09RG6LYM2,AH3OSZQWXXE6D4MJRKBAATPY7A4A,1483984240000,CoffeeSock ’The Original’ Reusable 100% Organi...,14.99,4.5,Amazon Home,4.4-4.5,long
5,Some key points about the product<br />- only ...,4.0,False,7,B09QC9947J,AG644AQBRT56BVFDCJMISCNTSRMA,1631115549428,"COMFEE' Countertop Dishwasher, Portable Dishwa...",349.95,4.4,Appliances,4.4-4.5,long
6,It’s very important that you know exactly what...,5.0,False,5,B0BWHKFM44,AG375WAXLZ7PIOQKIQ6KQB4J3JVQ,1626792779358,"VECYS Countertop Ice Maker Machine, 9 Bullet I...",99.90,4.4,Industrial & Scientific,4.4-4.5,long
7,We’ve been looking for a small freezer for 2 p...,5.0,False,3,B0BT8TKBY8,AHAR2ITQ3O2FJJDLIORAQ226KZHQ,1412864783000,Whynter CUF-301BK 3 cu. ft Energy Star Upright...,331.99,4.5,Appliances,4.4-4.5,long
8,"This is a good looking coffee drip system, and...",3.0,False,2,B09XL4P2HK,AHJQPUQLSQZE6LMIUMY7WNRXCQQQ,1654808215766,"Gocoffun Pour Over Coffee Dripper, Reusable Co...",20.99,4.5,Amazon Home,4.4-4.5,long
9,"OMG I love this thing, its mesmerizing to watc...",5.0,False,2,B077Y8DMHK,AEWPMZSG2PE52UCOYUD3J6ZYVQ2Q,1395381149000,Nina Soft Spin Portable Compact Spin Dryer (Bl...,NaN,4.4,Appliances,4.4-4.5,long
